# ML System Design Fundamentals — AI Lab Instructor Notebook

*System Design · Expert*



8 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
def section(title: str):
    print("\n" + "="*80)
    print(title)
    print("="*80)

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')


## Task 1: Dataset & Setup

# ML System Design Fundamentals — Student Lab

Case study: Fraud detection for card-not-present transactions.

This notebook is a structured design doc exercise. Fill in the templates with concise bullets.

## Prompt (the interview question)

**Design an ML system to detect fraudulent card-not-present transactions in real time.**

You must address: requirements, data, modeling, serving, monitoring, and iteration.

## 0 — Problem framing

## 1 — Requirements (SLOs + constraints)

In [ ]:
section("0 — Problem framing")
problem = {
  'users': ['risk team', 'customer support', 'cardholders', 'merchants'],
  'decision': 'approve/decline/step-up-auth for each transaction',
  'what_is_fraud': 'unauthorized transaction leading to chargeback',
  'label_delay_days': 30,
}
print(problem)
check('has_users', len(problem['users']) > 0)

**Why this works.** # ML System Design Fundamentals — Instructor Lab

Case study: Fraud detection for card-not-present transactions.

This notebook is a structured design doc exercise. Fill in the templates with concise bullets.

## Prompt (the interview question)

**Design an ML system to detect fraudulent card-not-present transactions in real time.**

You must address: requirements, data, modeling, serving, monitoring, and iteration.

## 0 — Problem framing

## 1 — Requirements (SLOs + constraints)

## 2 — Metrics (business + model + guardrails)

## 3 — Data + labeling plan

## 4 — Modeling plan (baselines → iterations)

## 5 — Serving architecture (online + batch)

## 6 — Monitoring + iteration loop

## 7 — Final: tradeoffs (2 required)

---
## Submission Checklist
- Filled each section
- Included concrete SLOs + metrics
- Identified leakage risks
- 2+ tradeoffs with decisions

## Task 2: 1 — Requirements (SLOs + constraints)

## 1 — Requirements (SLOs + constraints)

**Hints**
- s:

In [ ]:
section("1 — Requirements")
requirements = {
  'latency_p99_ms': 50,
  'throughput_rps': 10000,
  'availability': '99.99%',
  'privacy_constraints': ['PCI compliance', 'no raw PAN storage in features', 'PII tokenization/minimization'],
  'failure_mode': 'if model or features are unavailable, fall back to rules + step-up auth for risky transactions',
}
print(requirements)
check('latency_set', requirements['latency_p99_ms'] is not None)

## Task 3: 2 — Metrics (business + model + guardrails)

## 2 — Metrics (business + model + guardrails)

In [ ]:
section("2 — Metrics")
metrics = {
  'business': {
    'fraud_loss_prevented': 'USD prevented per day/week',
    'chargeback_rate': 'chargebacks / transactions',
  },
  'model': {
    'pr_auc': 'primary offline metric under class imbalance',
    'recall_at_fpr': 'recall at a fixed false-positive rate such as 0.1%',
  },
  'guardrails': {
    'false_positive_budget': 'e.g. <= 0.2% of legitimate transactions declined',
    'customer_friction_rate': 'share of transactions sent to step-up authentication',
    'manual_review_capacity_per_day': 500,
  }
}
print(metrics)
check('has_business_metric', metrics['business']['fraud_loss_prevented'] is not None)

## Task 4: 3 — Data + labeling plan

## 3 — Data + labeling plan

In [ ]:
section("3 — Data + labeling")
data_plan = {
  'sources': [
    'transaction events (amount, MCC, merchant id, time)',
    'user/account history aggregates (velocity, past chargebacks, spend patterns)',
    'device fingerprint and IP / geo signals',
    'merchant risk signals',
    'rules engine decisions and manual review outcomes',
  ],
  'label_definition': 'fraud = chargeback confirmed as unauthorized within 30 days',
  'label_delay_days': problem['label_delay_days'],
  'leakage_risks': [
    'post-transaction signals like chargeback filing included as features',
    'future aggregates that look past decision time',
    'manual review outcomes used without accounting for model-induced bias',
  ],
  'split_strategy': 'time-based split so validation simulates future production data',
}
print(data_plan)
check('has_leakage_risks', len(data_plan['leakage_risks']) >= 3)

## Task 5: 4 — Modeling plan (baselines → iterations)

## 4 — Modeling plan (baselines → iterations)

**Hints**
- s: baseline could be rules or logistic regression; iterations could be GBDT; consider calibration.

In [ ]:
section("4 — Modeling plan")
modeling = {
  'baseline': 'rules + logistic regression on simple aggregates',
  'candidate_models': ['GBDT (LightGBM/XGBoost)', 'neural model only if added complexity is justified'],
  'calibration': 'Platt scaling or isotonic regression on time-split validation',
  'thresholding': 'set score cutoffs so the top-risk transactions fit the manual-review budget while still satisfying the false-positive budget',
  'explainability': 'reason codes from rules + top feature attributions for model decisions',
}
print(modeling)
check('has_baseline', modeling['baseline'] is not None)

## Task 6: 5 — Serving architecture (online + batch)

## 5 — Serving architecture (online + batch)

In [ ]:
section("5 — Serving architecture")
architecture = {
  'online_path': 'transaction -> fetch/compute real-time features -> score model -> apply thresholds/rules -> approve/decline/step-up -> log outcome',
  'feature_store': 'shared offline/online feature store or shared feature definitions to reduce training-serving skew',
  'model_hosting': 'stateless scorer service behind a load balancer with versioned model artifacts',
  'fallback': 'rules engine + cached/default features + step-up auth when model path is degraded',
  'batch_jobs': 'daily feature backfills, label joins, training set build, retraining, and offline evaluation',
}
print(architecture)
check('has_online_path', architecture['online_path'] is not None)

## Task 7: 6 — Monitoring + iteration loop

## 6 — Monitoring + iteration loop

In [ ]:
section("6 — Monitoring")
monitoring = {
  'data_drift': 'monitor PSI/KS, missingness, schema changes, and feature freshness on key inputs',
  'performance_drift': 'with delayed labels, track recall at fixed FPR, calibration drift, and decision distribution over time',
  'alerts': [
    'p99 latency above SLO for 5 minutes',
    'decline or step-up rate spikes outside expected range',
    'drift threshold exceeded on high-importance features',
  ],
  'rollback': 'route traffic back to previous model version and rely on rules/step-up while investigating',
  'retraining_cadence': 'weekly by default, faster if fraud patterns shift materially',
}
print(monitoring)
check('three_alerts', len(monitoring['alerts']) >= 3)

## Task 8: 7 — Final: tradeoffs (2 required)

## 7 — Final: tradeoffs (2 required)

In [ ]:
section("7 — Tradeoffs")
tradeoffs = [
    'Latency vs feature richness: keep the online path under 50 ms by using only low-latency features online and pushing heavy aggregates to batch/nearline computation.',
    'False positives vs fraud loss: set thresholds to catch more fraud without overwhelming manual review or harming too many legitimate customers.',
    'Freshness vs stability: retrain frequently enough to catch new fraud patterns, but keep a rollback path and validation gates to avoid unstable deployments.',
]
print(tradeoffs)
check('two_tradeoffs', len(tradeoffs) >= 2)

In [ ]:
# Checks
check('two_tradeoffs', len(tradeoffs) >= 2)